# Current pipeline walkthrough (offline by default)

Run with a Python **3.10+ kernel** in which `intention-grounding` is installed. From the repository root, the existing environment runs the pipeline with `./.venv/bin/python`; Jupyter is optional and is not present in this checkout's Python 3.10 `.venv`; the parent Jupyter kernel uses unsupported Python 3.9. To enable the existing repository environment, run `uv pip install --python .venv/bin/python -e '.[notebook]'` from the root, then launch with `./.venv/bin/python -m notebook notebooks/pipeline_walkthrough.ipynb`. Launch from the repository root or `notebooks/`. Execute cells in order.

The interpreter and grounder replay responses are **manually authored software-contract fixtures**, not model evaluation. The bundled four-row CSV is **synthetic**. No WikiDBs-10k table files were found in this checkout; the WikiDBs metadata is a future-source record, not local data. The current pipeline has no verified gold intended-action or table-suitability labels for WikiDBs.


In [ ]:
import sys
if sys.version_info < (3, 10):
    raise RuntimeError("Select a Python 3.10+ kernel with intention-grounding installed; see README.md")

import json
from importlib.resources import files
from pathlib import Path

from intention_grounding.cli import _offline_inputs
from intention_grounding.interpreter.core import interpret
from intention_grounding.action_base import load_action_base, ancestry
from intention_grounding.grounder import ground
from intention_grounding.profiler import profile_table
from intention_grounding.aligner import compare_card

root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "pyproject.toml").is_file()), None)
if root is None:
    raise RuntimeError("Launch from the repository root or notebooks/")
print("Repository:", root)
print("Python:", sys.version.split()[0], "executable:", sys.executable)


## 1. Input request

The CLI's offline helper selects the `observations` case from the installed replay fixture. Inspect the actual input before calling the provider.


In [ ]:
input_id, request, interpreter_provider, grounder_provider = _offline_inputs()
print({"input_id": input_id, "request": request})


## 2. Interpreter: raw response and validated interpretation

The provider is a local replay object. Its response is never fetched from a network. The raw JSON is distinct from the interpreter's validated envelope.


In [ ]:
raw_interpreter = interpreter_provider.invoke(input_id, request)
raw_doc = json.loads(raw_interpreter)
print({"raw_response_bytes": len(raw_interpreter.encode("utf-8")),
       "raw_status": raw_doc["status"],
       "raw_actions": [a["id"] for a in raw_doc["actions"]],
       "raw_evidence": [(e["id"], e["text"]) for e in raw_doc["evidence"]]})


In [ ]:
interpreted = interpret(input_id, request, interpreter_provider)
assert interpreted["failure"] is None
print({"provider": interpreted["metadata"]["provider"],
       "failure": interpreted["failure"],
       "validated_status": interpreted["interpretation"]["status"],
       "validated_roles": [a["role"] for a in interpreted["interpretation"]["actions"][0]["arguments"]]})


## 3. Canonical Action Base

The loader checks references and parent links. The draft AB supports the action hierarchy below; the `kb/` knowledge base is from an older workflow.


In [ ]:
ab = load_action_base()
print("AB version:", ab["ab_version"])
for action_id, action in ab["actions"].items():
    print(action_id, "parent:", action["parent"], "abstract:", action.get("abstract"))
print("Selected action ancestry:", ancestry("compare.time", ab))


## 4. Grounder: proposal, validation, card

The proposed action and argument references come from a local replay response. `ground` independently validates the action ID, AB version, inherited roles, cardinality, and evidence references before constructing a card. A grounded card says the request was represented in the AB; it does not say any table is suitable.


In [ ]:
raw_grounder = grounder_provider.invoke(interpreted["interpretation"], ab)
print("Raw proposal:", json.loads(raw_grounder))


In [ ]:
grounded = ground(interpreted, ab, grounder_provider, run_id="notebook-offline")
assert grounded["status"] == "grounded", grounded["reasons"]
card = grounded["grounded_card"]
print({"status": grounded["status"],
       "validation": card["deterministic_validation"],
       "selected_action": card["selected_child_action_id"],
       "policy": card["bindings"]["coverage_policy"],
       "requirements": [(r["id"], r["axis"], r["predicate"]) for r in card["requirements"]]})


## 5. Local table profile and provenance

The profiler scans the entire bundled CSV, records observed statistics, and attaches file SHA-256 evidence. The source metadata labels this table as a synthetic fixture.


In [ ]:
table_path = Path(files("intention_grounding").joinpath("fixtures/offline_table.csv"))
metadata = json.loads(files("intention_grounding").joinpath("fixtures/offline-source-metadata.json").read_text())
profile = profile_table(table_path, dataset_id=metadata["dataset_id"], source_metadata=metadata)
print({"dataset": metadata["dataset_name"], "table_id": profile["table_id"],
       "rows": profile["row_count"], "sha256": profile["file_sha256"],
       "columns": [{k: c.get(k) for k in ("name", "observed_type", "null_count", "numeric_statistics", "distinct_count")}
                   for c in profile["columns"]],
       "evidence": profile["evidence"][0],
       "semantic_annotations": profile["semantic_annotations"]})


## 6. Axis comparison

The current profiler does not establish semantic meanings, units, populations, or observation relations required by the AB predicates. The aligner returns `unknown` for these unsupported checks. This is **not** a successful alignment or suitability finding.


In [ ]:
comparison = compare_card(card, [profile])
print({"policy": comparison["policy"],
       "table_state": comparison["tables"][0]["overall_state"],
       "requirements": [(r["axis"], r["predicate"], r["state"], r["reason"])
                        for r in comparison["tables"][0]["requirements"]]})


## 7. CLI artifact layout

Exact offline command from the repository root:

```bash
./.venv/bin/python -m intention_grounding.cli --mode offline --output outputs/pipeline
```

The next cell runs the same CLI in a temporary output directory and lists the artifact names. The CLI normally creates a unique `run-*` directory under the requested `--output` path. It makes no network calls in offline mode.


In [ ]:
import subprocess
import tempfile

with tempfile.TemporaryDirectory(prefix="intention-walkthrough-") as temp_dir:
    result = subprocess.run(
        [sys.executable, "-m", "intention_grounding.cli", "--mode", "offline",
         "--output", temp_dir],
        cwd=root, text=True, capture_output=True, check=True)
    runs = list(Path(temp_dir).glob("run-*"))
    assert len(runs) == 1
    manifest = json.loads((runs[0] / "run.json").read_text())
    print(result.stdout.strip())
    print("Artifact files:", sorted(p.name for p in runs[0].iterdir()))
    print("Manifest mode / AB version:", manifest["mode"], manifest["ab_version"])


## 8. Optional live Groq call — explicit opt-in

This cell is skipped by default. To opt in, configure `GROQ_API_KEY` and `GROQ_MODEL_ID` in the process environment, install the optional Groq SDK, and deliberately set `RUN_LIVE_GROQ = True` below. Use `./.venv/bin/python -m intention_grounding.cli --list-models` to list model IDs available to your account. The cell checks that both variables are present before making either model call and never displays the key. Live calls may incur charges.


In [ ]:
RUN_LIVE_GROQ = False

if RUN_LIVE_GROQ:
    import os
    from intention_grounding.interpreter.providers import GroqProvider
    from intention_grounding.grounder import GroqGrounderProvider

    model_id = os.environ.get("GROQ_MODEL_ID", "").strip()
    missing = []
    if not model_id:
        missing.append("GROQ_MODEL_ID")
    if not os.environ.get("GROQ_API_KEY"):
        missing.append("GROQ_API_KEY")
    if missing:
        raise RuntimeError("Set these process environment variables before a live run: " + ", ".join(missing))

    live_interpretation = interpret(input_id, request, GroqProvider(model=model_id))
    live_grounding = ground(live_interpretation, ab, GroqGrounderProvider(model=model_id),
                            run_id="notebook-live")
    print({"interpretation_failure": live_interpretation["failure"],
           "interpretation_provider_error": live_interpretation["provider_error"],
           "interpretation_status": (live_interpretation["interpretation"] or {}).get("status"),
           "grounding_status": live_grounding["status"],
           "grounding_provider_error": live_grounding["provider_error"],
           "grounding_reasons": live_grounding["reasons"]})
else:
    print("Skipped live Groq calls.")
